# Predict customer orders with synthetic TPC-H tables on Databricks

How many orders will a recently active customer place in the next 90 days? Train Kumo Engine on synthetic customer–order–line-item–part history, then score newly added customers. Binary classification and order-to-part ranking are optional alternatives. Synthetic metrics are demonstration only.


## Prepare your environment

Use Databricks GPU notebook compute with Python 3.12, supported local SSD scratch and enough RAM for the graph. Follow [Databricks setup](https://docs.nvidia.com/sdgm/kumo-engine/get-started-with-kumo-engine/installation/databricks-notebook). The SQL warehouse accesses Unity Catalog; model training uses notebook compute. Allow space for graph materialization and model/prediction files. `/tmp` is temporary; retain artifacts in a volume when needed.

## Install

Install Kumo Engine, then restart Python.

In [ ]:
%pip install "kumo-engine[databricks]"

In [ ]:
dbutils.library.restartPython()

<br>

## Enter your configuration

Choose your warehouse and an existing schema that you own for the generated tables and unique prediction output. The scoring date follows the generated history.

In [ ]:
from uuid import uuid4


from datetime import datetime
from pathlib import Path

# ── Databricks connector ──
CONNECTOR_CATALOG = "<connector_catalog>"
CONNECTOR_SCHEMA = "<connector_schema>"
SQL_WAREHOUSE_ID = "<sql_warehouse_id>"               # Starter Warehouse

# MODEL_NAME = "TPCH_CUSTOMER_ORDER_MODEL"
PREDICTION_TIME = datetime(2026, 1, 5)              # Generated history spans 2023 through 2025.
PREDICTION_OUTPUT_TABLE = ("TPCH_ORDER_PREDICTIONS") + "_" + uuid4().hex[:8]

# Durable prediction output.
WRITE_PREDICTIONS = True

# Use a schema containing only your generated example tables.


<br>

<a id="runtime"></a>
## Import and configure scratch

After each kernel restart, run configuration and verify that the runtime imports and a CUDA operation succeed. They do not depend on variables created during installation. 

In [ ]:
import shutil
import tempfile

from IPython.display import display
import torch
import torch_geometric

import kumo.engine as kumo
from kumo.engine import PredictionOutputConfig
from kumoapi.graph import Edge
from kumoapi.jobs import PredictionArtifactType, WriteMode
from kumoapi.task import TaskType
from kumo_databricks.output import SEPARATOR as _SEP


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("No GPU is available to this notebook.")

values = torch.tensor([1.0, 2.0], device="cuda")
groups = torch.tensor([0, 0], device="cuda")
total = torch.zeros(1, device="cuda").scatter_add_(0, groups, values)
assert total.item() == 3.0

print("Kumo:", kumo.__version__)
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
print("PyG:", torch_geometric.__version__)
print("GPU:", torch.cuda.get_device_name(0))

Configure scratch before constructing any Kumo table, graph, or trainer.

In [ ]:
SCRATCH_DIR = Path("/tmp/kumoai-local")
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)
with tempfile.NamedTemporaryFile(dir=SCRATCH_DIR) as probe:
    probe.write(b"scratch check")
    probe.flush()
kumo.configure(scratch_dir=str(SCRATCH_DIR))
print("Free scratch space (GiB):", round(shutil.disk_usage(SCRATCH_DIR).free / 2**30, 1))


<br>

## Generate synthetic example data

Run the next cell to create initial history and reserve new customers with their observed orders. You can skim the generation code; the graph workflow starts below. Load only `initial/` files now, preserving lowercase table/column names. Reserved rows are appended after fitting; the generator creates local files only.


In [ ]:
from pathlib import Path
import tempfile
import numpy as np
import pandas as pd

def make_tpch(seed, customer_start, customer_count, order_start, rows, start, days):
    rng = np.random.default_rng(seed)
    customer_ids = np.arange(customer_start, customer_start + customer_count)
    part_ids = np.arange(80)
    customer = rng.choice(customer_ids, rows)
    day = rng.integers(0, days, rows)
    keep = (customer % 4 != 0) | (day < int(days * 0.6))
    customer, day = customer[keep], day[keep]
    item = customer % 4 * 20 + rng.integers(20, size=len(day))
    dates = pd.Timestamp(start) + pd.to_timedelta(day, unit="D")
    amount = np.round(10 + item * 1.5 + rng.normal(0, 2, len(day)), 2)
    ids = np.arange(order_start, order_start + len(day))
    return {
        "customer": pd.DataFrame(dict(c_custkey=customer_ids, c_name=[f"Synthetic customer {i}" for i in customer_ids], c_nationkey=customer_ids % 5, c_mktsegment=customer_ids % 4, c_acctbal=100 + customer_ids)),
        "part": pd.DataFrame(dict(p_partkey=part_ids, p_name=[f"Synthetic part {i}" for i in part_ids], p_brand=part_ids % 5, p_size=1 + part_ids % 10, p_container="BOX", p_retailprice=20 + part_ids * 2, p_type=part_ids // 20)),
        "orders": pd.DataFrame(dict(o_orderkey=ids, o_custkey=customer, o_orderdate=dates, o_totalprice=amount, o_shippriority=customer % 3, o_orderstatus=np.where(customer % 2, "F", "O"), o_orderpriority=np.where(customer % 2, "HIGH", "LOW"))),
        "lineitem": pd.DataFrame(dict(lineitem_id=ids, l_shipmode="TRUCK", l_orderkey=ids, l_partkey=item, l_suppkey=item % 10, l_linenumber=1, l_quantity=1 + customer % 4, l_extendedprice=amount, l_discount=0.05, l_tax=0.02, l_shipdate=dates + pd.Timedelta(days=2), l_commitdate=dates + pd.Timedelta(days=1), l_receiptdate=dates + pd.Timedelta(days=3), l_returnflag="N", l_linestatus="O")),
    }

frames = make_tpch(42, 0, 200, 0, 5000, "2023-01-01", 1095)
reserved = make_tpch(43, 200, 20, 1_000_000, 600,
                     pd.Timestamp(PREDICTION_TIME) - pd.Timedelta(days=180), 177)
reserved.pop("part")  # New customers reuse the original catalog.
assert set(reserved["customer"].c_custkey).isdisjoint(frames["customer"].c_custkey)
assert set(reserved["orders"].o_orderkey).isdisjoint(frames["orders"].o_orderkey)
assert reserved["orders"].o_custkey.isin(reserved["customer"].c_custkey).all()
assert set(reserved["orders"].o_custkey) == set(reserved["customer"].c_custkey)
assert reserved["lineitem"].l_orderkey.isin(reserved["orders"].o_orderkey).all()
assert reserved["lineitem"].l_partkey.isin(frames["part"].p_partkey).all()
assert reserved["lineitem"].l_shipdate.lt(pd.Timestamp(PREDICTION_TIME)).all()

DATA_DIR = Path(tempfile.mkdtemp(prefix="synthetic-tpch-"))
initial_dir = DATA_DIR / "initial"
initial_dir.mkdir()
for name, frame in frames.items():
    frame.to_parquet(initial_dir / f"{name}.parquet", index=False)
print("Load only these initial files before training:", initial_dir)
print({name: len(frame) for name, frame in frames.items()})


<br>

## Connect to your loaded tables

Manually load each `initial/` Parquet file into the configured Unity Catalog schema under its filename stem. Preserve numeric keys and dates. The Databricks connector uses the notebook’s authentication context.


In [ ]:
connector = kumo.DatabricksConnector.from_notebook(
    catalog=CONNECTOR_CATALOG,
    schema_name=CONNECTOR_SCHEMA,
    warehouse_id=SQL_WAREHOUSE_ID,
)
print("Tables:", connector.table_names())


<br>

## Describe the four tables

A primary key identifies a row. A physical type describes its stored representation, such as a date; a semantic type describes its modeling role, such as a category or numerical value. 

The customer and part definitions select the columns used in this example. 

In [ ]:
customer = (
    kumo.Table.from_source_table(
        connector["customer"],
        ["c_custkey", "c_nationkey", "c_mktsegment", "c_acctbal"],
        primary_key="c_custkey",
    )
)

part = (
    kumo.Table.from_source_table(
        connector["part"],
        ["p_partkey", "p_name", "p_brand", "p_type", "p_size",
         "p_container", "p_retailprice"],
        primary_key="p_partkey",
    )
)

Event timestamps let Kumo separate information available before a prediction boundary from future outcomes. Using future information as an input causes temporal leakage.

This sample uses `o_orderdate` for orders and `l_shipdate` for line items. The ship date represents when the item became observable in the fulfillment process and is a reasonable availability timestamp for this dataset.

In [ ]:
orders = (
    kumo.Table.from_source_table(
        connector["orders"],
        ["o_orderkey", "o_custkey", "o_orderdate", "o_orderstatus",
         "o_totalprice", "o_orderpriority", "o_shippriority"],
        primary_key="o_orderkey",
        time_column="o_orderdate",
    )
)

In [ ]:
lineitem = (
    kumo.Table.from_source_table(
        connector["lineitem"],
        ["lineitem_id", "l_orderkey", "l_partkey", "l_shipdate",
         "l_quantity", "l_extendedprice", "l_discount", "l_tax",
         "l_returnflag", "l_linestatus", "l_shipmode"],
        primary_key="lineitem_id",
        time_column="l_shipdate",
    )
)

<br>

## Connect the tables and materialize a snapshot

A **graph** describes the tables and their relationships. Each arrow below goes from a foreign-key column to the destination table's primary key:

```text
lineitem.l_partkey   -> part.p_partkey
lineitem.l_orderkey  -> orders.o_orderkey
orders.o_custkey     -> customer.c_custkey
```

Validation checks the graph definition; it is not a blanket guarantee of source-data quality. A **snapshot** materializes source data into the local graph index used for training and prediction. Snapshot creation reads the tables and consumes local disk. Reuse requires an unchanged, valid retained snapshot; loading a model does not restore a lost index.

In [ ]:
graph = kumo.Graph(
    tables=[customer, part, orders, lineitem],
    edges=[
        Edge(src_table="orders", fkey="o_custkey", dst_table="customer"),
        Edge(src_table="lineitem", fkey="l_orderkey", dst_table="orders"),
        Edge(src_table="lineitem", fkey="l_partkey", dst_table="part"),
    ],
)

graph.validate()

<br>

## Inspect the graph

Inspect the actual graph’s tables and relationships before materialization. The [graph visualization](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/set-up-a-graph/graphs) call is a commented preview because it is unavailable in the supported API.

In [ ]:
# graph.visualize(show_cols=True)

<br>

### Materialize the graph

Read the initial loaded tables into the training snapshot.


In [ ]:
snapshot = graph.snapshot()
print("Snapshot:", snapshot.id)
print("Local index:", snapshot.index_path)

<br>

## Define the prediction question

Predictive Query Language (PQL) is Kumo Engine's declarative syntax for defining a predictive modeling task: it specifies the target, the entity to score, and the data context. This default query predicts how many orders a customer active in the past year will place in the next 90 days.

In [ ]:
# Regression: predict how many orders a customer active in the past year will place in the next 90 days.
pquery = kumo.PredictiveQuery(
    graph=graph,
    query=(
        "PREDICT COUNT(orders.*, 0, 90, days) "
        "FOR EACH customer.c_custkey "
        "WHERE COUNT(orders.*, -365, 0, days) > 0"
    ),
)
pquery.validate()
print("Task type:", pquery.get_task_type())

<br>

### Optional task alternatives

The next two code cells are commented out. To try one, skip or comment out the default regression query cell above, uncomment exactly one alternative, and run it before training. Both define `pquery` and use the existing `graph`, `snapshot`, and downstream variables. Train a new model for the selected task. The link example ranks parts associated with orders, not customer-level recommendations.

In [ ]:
# Binary classification: predict whether a customer active in the past year will order again in the next 90 days.
# pquery = kumo.PredictiveQuery(
#     graph=graph,
#     query="""
#         PREDICT COUNT(orders.*, 0, 90, days) > 0
#         FOR EACH customer.c_custkey
#         WHERE COUNT(orders.*, -365, 0, days) > 0
#     """,
# )

# pquery.validate()
# print("Task type:", pquery.get_task_type())

In [ ]:
# Link prediction: rank up to five parts that may ship in the next 90 days for an order with recent line items.
# pquery = kumo.PredictiveQuery(
#     graph=graph,
#     query=(
#         "PREDICT LIST_DISTINCT(lineitem.l_partkey, 0, 90, days) "
#         "RANK TOP 5 FOR EACH orders.o_orderkey "
#         "WHERE COUNT(lineitem.*, -365, 0, days) > 0"
#     ),
# )
# pquery.validate()
# print("Task type:", pquery.get_task_type())

<br>

### Generate training rows

Use approximate chronological 70/15/15 splits so validation and test follow training. Anchors are 90 days apart over the most recent 900 days; the generation plan leaves the target window observable. Inspect actual split dates and row counts (`0` train, `1` validation, `2` test).

In [ ]:
from kumoapi.model_plan import TrainingTableGenerationPlan
from kumo.engine.trainer.output_config import TrainingTableOutputConfig

generation_plan = TrainingTableGenerationPlan(
    split="ApproxDateOffsetSplit([0.7, 0.15, 0.15])",
    train_start_offset=900, train_end_offset=0, timeframe_step=90,
)
train_table = pquery.generate_training_table(snapshot, plan=generation_plan)
local_artifacts = DATA_DIR / "artifacts"
local_artifacts.mkdir(exist_ok=True)
local_output = kumo.FileConnector(root_dir=str(local_artifacts))
train_table.export(TrainingTableOutputConfig(connector=local_output, table_name="training_rows/"))
training_rows = pd.read_parquet(local_artifacts / "training_rows")
print(training_rows.groupby("SPLIT").agg(rows=("ENTITY1", "size"), first_anchor=("TIMESTAMP", "min"), last_anchor=("TIMESTAMP", "max")))


<br>

## Configure the model

The suggested model plan controls architecture, sampling, and optimization. The model plan is fully customizable: inspect and adjust its settings for your task, then create `trainer = kumo.Trainer(model_plan)` before fitting.

In [ ]:
from kumoapi.model_plan import RunMode

model_plan = pquery.suggest_model_plan(snapshot, run_mode=RunMode.FAST)
# Create a Trainer with your model plan.
model_plan.training_job.refit_trainval = False
model_plan.training_job.refit_full = False
trainer = kumo.Trainer(model_plan)
print(model_plan)


<br>

### Train the model

After creating a Trainer from the model plan, call `fit()` with the training table and compatible graph snapshot.


In [ ]:
# Serverless GPU workaround for Optuna child process IPC authentication errors.
# import multiprocessing
# try:
#     multiprocessing.set_start_method("spawn", force=True)
# except RuntimeError:
#     pass

training_result = trainer.fit(train_table, snapshot=snapshot)


<br>

### Evaluate

**Synthetic metrics — demonstration only.** For the default regression task, MAE/RMSE measure order-count error; lower is better. For binary classification, AUROC measures repeat-order ranking. For the link alternative, ranking metrics reward relevant parts near the top. Inspect each metric’s validation/test stage; use validation for choices and reserve test for assessment. Retain the complete model bundle before the runtime ends.


In [ ]:
print("Synthetic metrics — demonstration only:", training_result.metrics)
print("Reusable local bundle:", training_result.bundle_dir)
for name, path in training_result.artifact_paths.items():
    print(name, path)


<br>

### Check the source dates

Inspect the modeled orders table before adding the new scoring records.


In [ ]:
date_range = spark.sql(f"""
    SELECT MIN(o_orderdate) AS first_order, MAX(o_orderdate) AS last_order
    FROM {CONNECTOR_CATALOG}.{CONNECTOR_SCHEMA}.orders
""")
display(date_range.toPandas())
print("Configured prediction boundary:", PREDICTION_TIME)

<br>

## Batch predictions

Append the reserved new customers, orders and line items to the same source tables after training. The shared part catalog stays unchanged. All reserved events precede the scoring anchor. The original training snapshot remains available; a fresh connector and snapshot read the updated sources.

In [ ]:
append_dir = DATA_DIR / "append"
append_dir.mkdir(exist_ok=False)
for name, frame in reserved.items():
    frame.to_parquet(append_dir / f"{name}.parquet", index=False)
print("Append these files once to the matching source tables:", append_dir)
print({name: len(frame) for name, frame in reserved.items()})

<br>

### Load and verify the appended rows

Manually append each `append/` file to its matching source table, preserving existing rows. Append exactly once; do not reload `initial/`, replace tables, or append the part catalog. Continue only after the upload completes. These checks compare the loaded data with the expected original-plus-new tables.


In [ ]:
# Verify that manual appends added exactly the reserved rows, with unique keys.
keys = {'customer': 'c_custkey', 'part': 'p_partkey', 'orders': 'o_orderkey', 'lineitem': 'lineitem_id'}
for name, initial in frames.items():
    loaded = spark.table(f"{CONNECTOR_CATALOG}.{CONNECTOR_SCHEMA}.{name}").toPandas()
    expected = pd.concat([initial, reserved[name]], ignore_index=True) if name in reserved else initial
    key = keys[name]
    assert loaded[key].is_unique, f"Duplicate keys in {name}; do not append twice."
    assert len(loaded) == len(expected), f"Complete the one-time append for {name}."
    assert set(loaded[key]) == set(expected[key]), f"Unexpected IDs in {name}."
    pd.testing.assert_frame_equal(
        loaded[expected.columns].sort_values(key).reset_index(drop=True),
        expected.sort_values(key).reset_index(drop=True),
        check_dtype=False, check_exact=False,
    )


<br>

### Prepare the new-record population

Rebuild the same graph definitions through a fresh connector, then prepare only the reserved entity IDs with their scoring anchor. No future outcomes are supplied.


In [ ]:
from dataclasses import replace

scoring_connector = kumo.DatabricksConnector.from_notebook(
    catalog=CONNECTOR_CATALOG, schema_name=CONNECTOR_SCHEMA, warehouse_id=SQL_WAREHOUSE_ID)
scoring_graph = kumo.Graph(
    tables=[replace(table, connector=scoring_connector) for table in graph.tables],
    edges=graph.edges,
)
scoring_graph.validate()
scoring_snapshot = scoring_graph.snapshot()
assert scoring_snapshot.id != snapshot.id
scoring_query = kumo.PredictiveQuery(scoring_graph, pquery.query)
scoring_query.validate()
new_ids = (reserved["orders"]["o_orderkey"] if pquery.get_task_type().is_link_pred
           else reserved["customer"]["c_custkey"])
scoring_population = pd.DataFrame({"ENTITY1": new_ids, "TIMESTAMP": PREDICTION_TIME})
assert len(scoring_population) >= 2 and scoring_population.ENTITY1.is_unique
assert not any(name.startswith("TARGET") for name in scoring_population)
scoring_population.to_parquet(local_artifacts / "scoring_population.parquet", index=False)
prediction_table = kumo.PredictionTable.from_source_table(
    local_output["scoring_population"], pq=scoring_query, snapshot=scoring_snapshot,
    plan=trainer.training_table_generation_plan,
)
population = pd.read_parquet(prediction_table.table_path)
assert len(population) == len(scoring_population)
assert set(population.ENTITY1) == set(new_ids)
print(population.head(8).to_string(index=False))


<br>

### Configure prediction output

Default regression and binary classification score only the new customers. The optional link query instead scores only the new orders and returns five part candidates per order. Use the unique output destination configured for this run.

In [ ]:
# Specific threshold or candidate count are required for binary classification and link prediction tasks respectively. 
prediction_options = {}
task_type = pquery.get_task_type()
if task_type == TaskType.BINARY_CLASSIFICATION:
    prediction_options["binary_classification_threshold"] = 0.5
elif task_type.is_link_pred:
    prediction_options["num_classes_to_return"] = 5

output_name = f"{CONNECTOR_SCHEMA}{_SEP}{PREDICTION_OUTPUT_TABLE}"

output_config = None
if WRITE_PREDICTIONS:
    output_config = PredictionOutputConfig(
        connector=connector,
        artifact_type=PredictionArtifactType.PREDICTIONS,
        table_name=output_name,
        write_mode=WriteMode.OVERWRITE,
    )

<br>

### Score the new records

Apply the fitted model to the updated snapshot without retraining.


In [ ]:
result = trainer.predict(
    prediction_table,
    snapshot=scoring_snapshot,
    output_config=output_config, **prediction_options,
)


<br>

### Inspect and save results

Verify IDs against the reserved scoring population. Regression returns one order-count prediction per customer; binary classification returns its probability and decision. Link prediction returns five ranked part rows per order. Save the full small batch and inspect ten rows.

In [ ]:
import pandas as pd
from pathlib import Path
from uuid import uuid4

population = pd.read_parquet(prediction_table.table_path)
assert len(population) >= 2, "Choose at least two scoring rows."
scoring_rows = population[[name for name in ("ENTITY1", "TIMESTAMP") if name in population]]
assert not any(str(column).startswith("TARGET") for column in scoring_rows)
batch_scores = result.frame
assert set(batch_scores["ENTITY"]) == set(population["ENTITY1"])
if pquery.get_task_type().is_link_pred:
    assert batch_scores.groupby("ENTITY").size().eq(5).all()
    assert not batch_scores.duplicated(["ENTITY", "CLASS"]).any()
else:
    assert len(batch_scores) == len(population)
    assert batch_scores.ENTITY.is_unique
print(batch_scores.head(10).to_string(index=False))
saved_predictions = Path.cwd() / ("batch-predictions-" + uuid4().hex + ".parquet")
batch_scores.to_parquet(saved_predictions, index=False)
print("Saved:", saved_predictions)

<br>

## Retain and reuse

Retain the complete model bundle and prediction export in durable storage. Rebuild the compatible source graph and scoring snapshot in a later session, then load the bundle with `kumo.Trainer.load()`; the model does not contain source data. See [model reuse](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/publish-and-retrieve-models).
